# LeetCode #79 — Word Search: DFS + backtracking trên lưới

**Mức độ:** Medium · **Chủ đề:** Array, String, Backtracking, Depth-First Search, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
```

Mục tiêu của notebook:

1. DFS từ mỗi ô, đi theo 4 hướng, khớp từng ký tự của `word`.
2. **Đánh dấu ô đã dùng** trên đường hiện tại và **gỡ đánh dấu** khi quay lui.
3. Phân biệt "đã thăm trong đường hiện tại" (backtracking) với "đã thăm toàn cục" (BFS/DFS thường) — dùng nhầm là sai.
4. Cắt tỉa: đếm ký tự, và tìm theo chiều ngược khi chữ cuối hiếm hơn.

# 1. Bài toán là gì?

Cho lưới ký tự `board` kích thước `m × n` và chuỗi `word`. Trả về `True` nếu `word` có trong lưới.

Từ được tạo bằng các ô **kề nhau** theo chiều ngang hoặc dọc. **Mỗi ô dùng tối đa một lần** trong một từ.

**Constraints (theo đề):**

- `m == board.length`, `n == board[i].length`
- `1 <= m, n <= 6`
- `1 <= word.length <= 15`
- `board` và `word` chỉ gồm chữ cái tiếng Anh in hoa và in thường.

**Follow up (theo đề):** có thể dùng **cắt tỉa tìm kiếm** để lời giải chạy nhanh hơn với `board` lớn hơn không?

---

## Ví dụ

```text
board = A B C E
        S F C S
        A D E E

word = "ABCCED"  -> True
word = "SEE"     -> True
word = "ABCB"    -> False   (ô 'B' không được dùng lại)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Duyệt theo chiều sâu (DFS — Depth-First Search)
2. Quay lui (backtracking)
3. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [1]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [2]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [3]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [4]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.2 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [5]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [6]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [7]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [8]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]:

 ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [9]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.3 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [10]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [11]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [12]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [13]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.4 Áp dụng vào bài này

Tìm đường đi trên **lưới** theo các ô kề (mảng hướng 4 chiều) ghép thành từ → **DFS** từ mỗi ô, kết hợp **quay lui**: đánh dấu ô đang dùng trên đường đi, thử 4 hướng, rồi **bỏ đánh dấu** khi quay lên để đường đi khác dùng được ô đó.

# 3. Nền tảng: duyệt 4 hướng trên lưới

Dùng danh sách hướng `DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]` và kiểm tra **trong lưới** trước khi truy cập:

In [14]:
from typing import List
import random

DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]

def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3, 4)))
print(list(neighbors(1, 1, 3, 4)))

[(0, 1), (1, 0)]


[(1, 2), (2, 1), (1, 0), (0, 1)]


# 4. Ý tưởng chính: DFS với quay lui

`dfs(r, c, k)` trả `True` nếu có thể khớp `word[k:]` bắt đầu tại ô `(r, c)`:

```text
dfs(r, c, k):
    nếu (r, c) ngoài lưới hoặc board[r][c] != word[k]: return False
    nếu k == len(word) - 1: return True        # đã khớp ký tự cuối
    đánh dấu (r, c) đã dùng
    kết quả = dfs ở bất kỳ ô kề nào với k + 1
    gỡ đánh dấu (r, c)                          # QUAY LUI
    return kết quả
```

Mẹo đánh dấu **không tốn bộ nhớ**: tạm gán `board[r][c] = '#'` (ký tự không thể có trong `word`), rồi trả lại giá trị cũ.

In [15]:
def exist_dfs(board: List[List[str]], word: str) -> bool:
    m, n = len(board), len(board[0])

    def dfs(r, c, k):
        if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        saved, board[r][c] = board[r][c], "#"
        found = any(dfs(r + dr, c + dc, k + 1) for dr, dc in DIRS)
        board[r][c] = saved
        return found

    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

board = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
for w in ["ABCCED", "SEE", "ABCB"]:
    print(w, exist_dfs(board, w))

ABCCED True
SEE True
ABCB False


`any(generator)` dừng ngay khi gặp `True` đầu tiên (short-circuit) — nhưng vì `board[r][c] = saved` nằm **sau** `any`, ô vẫn luôn được trả lại.

# 5. Mô phỏng

In [16]:
def trace_exist(board: List[List[str]], word: str) -> bool:
    board = [row[:] for row in board]
    m, n = len(board), len(board[0])

    def dfs(r, c, k, depth):
        ind = "  " * depth
        if not (0 <= r < m and 0 <= c < n):
            return False
        if board[r][c] != word[k]:
            if depth > 0 and board[r][c] != "#":
                print(f"{ind}({r},{c}) = {board[r][c]!r} ≠ {word[k]!r}")
            return False
        print(f"{ind}({r},{c}) = {word[k]!r} khớp word[{k}]")
        if k == len(word) - 1:
            print(f"{ind}=> tìm thấy!")
            return True
        saved, board[r][c] = board[r][c], "#"
        found = any(dfs(r + dr, c + dc, k + 1, depth + 1) for dr, dc in DIRS)
        board[r][c] = saved
        if not found:
            print(f"{ind}quay lui khỏi ({r},{c})")
        return found

    return any(dfs(r, c, 0, 0) for r in range(m) for c in range(n))

print(trace_exist(board, "ABCB"))

(0,0) = 'A' khớp word[0]
  (0,1) = 'B' khớp word[1]
    (0,2) = 'C' khớp word[2]
      (0,3) = 'E' ≠ 'B'
      (1,2) = 'C' ≠ 'B'
    quay lui khỏi (0,2)
    (1,1) = 'F' ≠ 'C'
  quay lui khỏi (0,1)
  (1,0) = 'S' ≠ 'B'
quay lui khỏi (0,0)
(2,0) = 'A' khớp word[0]
  (2,1) = 'D' ≠ 'B'
  (1,0) = 'S' ≠ 'B'
quay lui khỏi (2,0)
False


Từ `C` ở `(0,2)`, ô `B` kề nó là `(0,1)` — nhưng `(0,1)` đang mang dấu `'#'` (đã dùng trên đường) nên không khớp. Không còn đường khác → `False`.

# 6. Cắt tỉa (follow-up)

1. **Đếm ký tự:** nếu `word` cần nhiều ký tự `x` hơn số `x` trong lưới → `False` ngay. Cũng loại nhanh khi `len(word) > m · n`.
2. **Đảo từ:** DFS mở rộng nhiều nhánh nếu ký tự **đầu** phổ biến. Nếu ký tự cuối hiếm hơn ký tự đầu, tìm `word[::-1]` (một đường đọc ngược vẫn là đường hợp lệ).

In [17]:
from collections import Counter

def exist_pruned(board: List[List[str]], word: str) -> bool:
    m, n = len(board), len(board[0])
    if len(word) > m * n:
        return False
    cnt = Counter(ch for row in board for ch in row)
    need = Counter(word)
    if any(cnt[ch] < k for ch, k in need.items()):
        return False
    if cnt[word[0]] > cnt[word[-1]]:
        word = word[::-1]
    return exist_dfs([row[:] for row in board], word)


def count_dfs_calls(board, word):
    calls = 0
    m, n = len(board), len(board[0])
    b = [row[:] for row in board]
    def dfs(r, c, k):
        nonlocal calls
        calls += 1
        if not (0 <= r < m and 0 <= c < n) or b[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        saved, b[r][c] = b[r][c], "#"
        found = any(dfs(r + dr, c + dc, k + 1) for dr, dc in DIRS)
        b[r][c] = saved
        return found
    any(dfs(r, c, 0) for r in range(m) for c in range(n))
    return calls

hard = [["A"] * 6 for _ in range(6)]
w = "A" * 12 + "B"
print("Số lời gọi dfs khi tìm 'AAA...AB':", count_dfs_calls(hard, w))
print("Số lời gọi dfs khi tìm ngược 'BAAA...A':", count_dfs_calls(hard, w[::-1]))
print("exist_pruned:", exist_pruned(hard, w), "(loại ngay nhờ đếm ký tự: lưới không có 'B')")

Số lời gọi dfs khi tìm 'AAA...AB': 2502164
Số lời gọi dfs khi tìm ngược 'BAAA...A': 36
exist_pruned: False (loại ngay nhờ đếm ký tự: lưới không có 'B')


Lưới toàn `'A'`: tìm `"AAA…AB"` phải thử **rất nhiều** đường gồm 12 chữ `A` rồi mới thất bại ở chữ `B`. Đảo từ thành `"BAA…A"`: không có `B` nào để bắt đầu → dừng ngay sau `m · n` lời gọi.

# 7. Phiên bản đúng format LeetCode

In [18]:
class Solution:
    def exist(self, board: List[List[str]], word: str) -> bool:
        m, n = len(board), len(board[0])
        if len(word) > m * n:
            return False
        cnt = Counter(ch for row in board for ch in row)
        if any(cnt[ch] < k for ch, k in Counter(word).items()):
            return False
        if cnt[word[0]] > cnt[word[-1]]:
            word = word[::-1]

        def dfs(r: int, c: int, k: int) -> bool:
            if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[k]:
                return False
            if k == len(word) - 1:
                return True
            saved, board[r][c] = board[r][c], "#"
            found = (dfs(r + 1, c, k + 1) or dfs(r - 1, c, k + 1)
                     or dfs(r, c + 1, k + 1) or dfs(r, c - 1, k + 1))
            board[r][c] = saved
            return found

        return any(dfs(r, c, 0) for r in range(m) for c in range(n))

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


def brute_exist(board, word):
    m, n = len(board), len(board[0])
    def go(r, c, k, used):
        if board[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        for nr, nc in neighbors(r, c, m, n):
            if (nr, nc) not in used and go(nr, nc, k + 1, used | {(nr, nc)}):
                return True
        return False
    return any(go(r, c, 0, {(r, c)}) for r in range(m) for c in range(n))


sol = Solution()
check("Ví dụ 1", sol.exist([row[:] for row in board], "ABCCED"), True)
check("Ví dụ 2", sol.exist([row[:] for row in board], "SEE"), True)
check("Ví dụ 3", sol.exist([row[:] for row in board], "ABCB"), False)
check("1 ô", sol.exist([["a"]], "a"), True)
check("Từ dài hơn lưới", sol.exist([["a", "b"]], "aba"), False)
check("Phân biệt hoa thường", sol.exist([["a", "B"]], "ab"), False)
check("Lưới toàn A, có B ở góc", sol.exist([["A"] * 5 + ["B"]] + [["A"] * 6 for _ in range(5)], "A" * 14 + "B"), True)

random.seed(79)
for _ in range(3000):
    m, n = random.randint(1, 4), random.randint(1, 4)
    bd = [[random.choice("ab") for _ in range(n)] for _ in range(m)]
    w = "".join(random.choice("ab") for _ in range(random.randint(1, 7)))
    snapshot = [row[:] for row in bd]
    assert sol.exist(bd, w) == brute_exist(snapshot, w), (snapshot, w)
    assert bd == snapshot                       # board được trả lại nguyên trạng
print("3000 lưới ngẫu nhiên: khớp brute force, và board không bị thay đổi")

Ví dụ 1                  PASS  True
Ví dụ 2                  PASS  True
Ví dụ 3                  PASS  False
1 ô                      PASS  True
Từ dài hơn lưới          PASS  False
Phân biệt hoa thường     PASS  False
Lưới toàn A, có B ở góc  PASS  True
3000 lưới ngẫu nhiên: khớp brute force, và board không bị thay đổi


# 8. Vì sao đúng? Độ phức tạp?

- DFS liệt kê **mọi** đường đi đơn (không lặp ô) bắt đầu tại mỗi ô, khớp tiền tố của `word` — dừng sớm khi ký tự không khớp.
- Dấu `'#'` đảm bảo một ô không dùng hai lần **trên cùng một đường**; việc trả lại giá trị khi quay lui đảm bảo **đường khác** vẫn được dùng ô đó.

Gọi `L = len(word)`. Ô đầu có 4 hướng, các bước sau tối đa 3 hướng (không quay lại ô vừa đi):

```text
Time:  O(m · n · 3^L)  (trường hợp xấu)
Space: O(L) cho đệ quy
```

Với `m, n <= 6`, `L <= 15`, cắt tỉa giúp tránh các trường hợp xấu như ví dụ "toàn A".

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên gỡ đánh dấu khi quay lui

In [20]:
def wrong_no_unmark(board: List[List[str]], word: str) -> bool:
    board = [row[:] for row in board]
    m, n = len(board), len(board[0])
    def dfs(r, c, k):
        if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        board[r][c] = "#"                         # SAI: không trả lại
        return any(dfs(r + dr, c + dc, k + 1) for dr, dc in DIRS)
    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

bd = [["B", "A", "A"]]
print("Sai :", wrong_no_unmark(bd, "AAB"), "| Đúng:", sol.exist([row[:] for row in bd], "AAB"))

Sai : False | Đúng: True


Lưới chỉ một hàng `B A A`, tìm `"AAB"`:

| Bước | Điều xảy ra (bản sai) |
|---|---|
| Bắt đầu ở `(0,1)` | khớp `A`, đánh dấu `#`; sang `(0,2)` khớp `A`, đánh dấu `#`; cần `B` kề `(0,2)` — không có → thất bại |
| Không gỡ dấu | `(0,1)` và `(0,2)` vẫn là `#` |
| Bắt đầu ở `(0,2)` | ô đã thành `#` ≠ `A` → bỏ qua — **mất** đường đúng `(0,2) → (0,1) → (0,0)` |

Một đường **thất bại** đã "đốt" các ô mà đường **thành công** về sau cần đến.

## Lỗi 2: Dùng tập `visited` **toàn cục** (như BFS thường)

In [21]:
def wrong_global_visited(board: List[List[str]], word: str) -> bool:
    m, n = len(board), len(board[0])
    visited = set()                                # SAI: không bao giờ bỏ ra
    def dfs(r, c, k):
        if not (0 <= r < m and 0 <= c < n) or (r, c) in visited or board[r][c] != word[k]:
            return False
        if k == len(word) - 1:
            return True
        visited.add((r, c))
        return any(dfs(r + dr, c + dc, k + 1) for dr, dc in DIRS)
    return any(dfs(r, c, 0) for r in range(m) for c in range(n))

print("Sai :", wrong_global_visited(bd, "AAB"), "| Đúng:", sol.exist([row[:] for row in bd], "AAB"))

Sai : False | Đúng: True


Đây cùng một lỗi với Lỗi 1, chỉ khác cách đánh dấu. Trong backtracking, "đã dùng" là thuộc tính của **đường hiện tại**, không phải của **cả quá trình tìm**.

## Lỗi 3: Kiểm tra `k == len(word)` sau khi truy cập `word[k]`

Nếu điều kiện dừng là `k == len(word)` nhưng lại so `board[r][c] != word[k]` **trước** → `IndexError`. Hoặc kiểm tra dừng trước, hoặc dừng ở `k == len(word) - 1` như trên.

## Lỗi 4: Không trả lại `board` khi tìm thấy

Nếu viết `if dfs(...): return True` mà **chưa** trả lại ô → board bị hỏng. LeetCode không kiểm tra, nhưng trong code thật đây là bug "tác dụng phụ".

# 10. Mẹo nhận diện pattern

**DFS + backtracking trên lưới** khi cần tìm **đường đi đơn** (không lặp ô) thỏa điều kiện:

- Đánh dấu khi vào, gỡ khi ra.
- Cắt tỉa sớm khi tiền tố không khớp.

So sánh: **#200 Number of Islands** dùng `visited` toàn cục (chỉ cần biết ô đã được tô chưa) — **không** quay lui.

Bài liên quan: **#212 Word Search II** (Hard, dùng Trie), **#1219 Path with Maximum Gold**, **#980 Unique Paths III** (Hard).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `board` ở ví dụ, tìm đường cho `"SFCS"` và `"ASADFB"`. Từ nào có, từ nào không?

## Bài 2 — Điền chỗ trống

In [22]:
def exercise_2_exist(board: List[List[str]], word: str) -> bool:
    m, n = len(board), len(board[0])
    def dfs(r, c, k):
        # TODO: kiểm tra biên + khớp ký tự, điều kiện dừng, đánh dấu, đệ quy 4 hướng, gỡ dấu
        pass
    return any(dfs(r, c, 0) for r in range(m) for c in range(n))


# B = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
# assert exercise_2_exist([r[:] for r in B], "ABCCED") is True
# assert exercise_2_exist([r[:] for r in B], "ABCB") is False

## Bài 3 — Biến thể: trả về đường đi

Trả về list tọa độ của một đường khớp `word`, hoặc `None`.

In [23]:
def exercise_3_find_path(board: List[List[str]], word: str):
    # TODO
    pass


# B = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
# assert exercise_3_find_path([r[:] for r in B], "SEE") == [(1, 3), (2, 3), (2, 2)]
# assert exercise_3_find_path([r[:] for r in B], "ABCB") is None

## Bài 4 — Biến thể: Path with Maximum Gold (LeetCode #1219)

Mỗi ô là số vàng (`0` = không đi vào được). Bắt đầu ở ô bất kỳ có vàng, đi 4 hướng, không lặp ô, không vào ô 0. Tối đa tổng vàng?

```python
get_max_gold([[0, 6, 0], [5, 8, 7], [0, 9, 0]]) -> 24      # 9 → 8 → 7
```

In [24]:
def exercise_4_get_max_gold(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_get_max_gold([[0, 6, 0], [5, 8, 7], [0, 9, 0]]) == 24
# assert exercise_4_get_max_gold([[1, 0, 7], [2, 0, 6], [3, 4, 5], [0, 3, 0], [9, 0, 20]]) == 28

## Bài 5 — Mini challenge: Word Search II (đơn giản hóa)

Cho danh sách `words`, trả về các từ có trong lưới. Thay vì gọi `exist` cho từng từ, hãy dựng **Trie** (cây tiền tố) và DFS một lần từ mỗi ô, đi theo Trie.

In [25]:
def exercise_5_find_words(board: List[List[str]], words: List[str]) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_5_find_words([["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]], ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
# assert exercise_5_find_words([["a", "b"], ["c", "d"]], ["abcb"]) == []

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_exist` sau khi tự làm.

**Bài 3**
- Mức 1: giữ list `path`; `append` khi vào ô, `pop` khi quay lui; khi khớp ký tự cuối, trả `path[:]`.

**Bài 4**
- Mức 1: `dfs(r, c)` trả số vàng lớn nhất từ `(r, c)`: `grid[r][c] + max(0, max các dfs hàng xóm)`.
- Mức 2: đánh dấu bằng cách tạm gán `grid[r][c] = 0`.

**Bài 5**
- Mức 1: Trie là dict lồng nhau; đánh dấu kết thúc từ bằng khóa `"$"` chứa chính từ đó.
- Mức 2: `dfs(r, c, node)`: nếu `board[r][c]` không có trong `node` thì dừng; ngược lại đi vào `node[ch]`, nếu có `"$"` thì thêm từ (và xóa `"$"` để không thêm trùng).

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
for w in ["SFCS", "ASADFB"]:
    print(w, "->", sol.exist([row[:] for row in board], w))

SFCS -> True
ASADFB -> True


Cả hai đều có. `"SFCS"`: `S(1,0) → F(1,1) → C(1,2) → S(1,3)`. `"ASADFB"`: `A(0,0) → S(1,0) → A(2,0) → D(2,1) → F(1,1) → B(0,1)` — một đường vòng qua cả hai chữ `A` ở cột 0. (Nếu bắt đầu từ `A(2,0)` thì tới `A(0,0)` sẽ không có `D` kề — cần quay lui thử điểm xuất phát khác.)

## Đáp án Bài 2

In [27]:
def answer_2_exist(board: List[List[str]], word: str) -> bool:
    return exist_dfs(board, word)

B = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
assert answer_2_exist([r[:] for r in B], "ABCCED") is True
assert answer_2_exist([r[:] for r in B], "ABCB") is False
print("Bài 2 PASS (giống exist_dfs ở mục 4)")

Bài 2 PASS (giống exist_dfs ở mục 4)


## Đáp án Bài 3

In [28]:
def answer_3_find_path(board: List[List[str]], word: str):
    m, n = len(board), len(board[0])
    path = []
    def dfs(r, c, k):
        if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[k]:
            return None
        path.append((r, c))
        if k == len(word) - 1:
            return path[:]
        saved, board[r][c] = board[r][c], "#"
        for dr, dc in DIRS:
            res = dfs(r + dr, c + dc, k + 1)
            if res:
                board[r][c] = saved
                return res
        board[r][c] = saved
        path.pop()
        return None
    for r in range(m):
        for c in range(n):
            res = dfs(r, c, 0)
            if res:
                return res
    return None

assert answer_3_find_path([r[:] for r in B], "SEE") == [(1, 3), (2, 3), (2, 2)]
assert answer_3_find_path([r[:] for r in B], "ABCB") is None
print(answer_3_find_path([r[:] for r in B], "ABCCED"))
print("Bài 3 PASS")

[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1)]
Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_get_max_gold(grid: List[List[int]]) -> int:
    m, n = len(grid), len(grid[0])
    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n) or grid[r][c] == 0:
            return 0
        gold, grid[r][c] = grid[r][c], 0
        best = max(dfs(r + dr, c + dc) for dr, dc in DIRS)
        grid[r][c] = gold
        return gold + best
    return max(dfs(r, c) for r in range(m) for c in range(n))

assert answer_4_get_max_gold([[0, 6, 0], [5, 8, 7], [0, 9, 0]]) == 24
assert answer_4_get_max_gold([[1, 0, 7], [2, 0, 6], [3, 4, 5], [0, 3, 0], [9, 0, 20]]) == 28
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_find_words(board: List[List[str]], words: List[str]) -> List[str]:
    trie = {}
    for w in words:
        node = trie
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = w
    m, n = len(board), len(board[0])
    found = []
    def dfs(r, c, node):
        ch = board[r][c]
        if ch not in node:
            return
        nxt = node[ch]
        if "$" in nxt:
            found.append(nxt.pop("$"))
        board[r][c] = "#"
        for nr, nc in neighbors(r, c, m, n):
            dfs(nr, nc, nxt)
        board[r][c] = ch
    for r in range(m):
        for c in range(n):
            dfs(r, c, trie)
    return found

Bw = [["o", "a", "a", "n"], ["e", "t", "a", "e"], ["i", "h", "k", "r"], ["i", "f", "l", "v"]]
assert sorted(answer_5_find_words([r[:] for r in Bw], ["oath", "pea", "eat", "rain"])) == ["eat", "oath"]
assert answer_5_find_words([["a", "b"], ["c", "d"]], ["abcb"]) == []
for _ in range(500):
    bd = [[random.choice("ab") for _ in range(3)] for _ in range(3)]
    ws = list({"".join(random.choice("ab") for _ in range(random.randint(1, 5))) for _ in range(6)})
    assert sorted(answer_5_find_words([r[:] for r in bd], ws)) == sorted(w for w in ws if brute_exist(bd, w))
print("Bài 5 PASS")

Bài 5 PASS


Trie giúp các từ **chung tiền tố** dùng chung một lần DFS — thay vì `len(words)` lần tìm kiếm riêng.

# 14. Tổng kết

> DFS từ mỗi ô; khớp `word[k]` thì đánh dấu ô, thử 4 hướng với `k + 1`, rồi **gỡ dấu**. "Đã dùng" là của **đường hiện tại**, không phải toàn cục. Cắt tỉa bằng đếm ký tự và tìm theo chiều ít nhánh hơn.

---

## Checklist trước khi nộp

- [ ] Kiểm tra biên trước khi truy cập `board[r][c]`?
- [ ] Gỡ đánh dấu ở **mọi** nhánh (kể cả khi tìm thấy)?
- [ ] Không dùng `visited` toàn cục?
- [ ] Có cắt tỉa (đếm ký tự / đảo từ)?